# Missingness sensitivity during drought

Compare spatial nearest-station reconstruction and linear time interpolation after removing
10%, 25%, and 40% of a complete target series during PDSI <= -2 windows. The empirical
5th/95th percentiles describe sensitivity to missing observations; they are not confidence
intervals for measured consumption.

The default run uses **synthetic hydrology and synthetic facility labels**. To run with
observations, set `ECOLLATERAL_RESEARCH_DIR` to a directory containing `stations.csv`,
`coordinates.csv`, `daily_features.csv`, and a trusted `model.joblib`. Observed station
series should use MGD; the complete target is needed as the reconstruction reference.

In [ ]:
from pathlib import Path
import os
import sys

root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src.demo import create_demo
from src.models.missingness import analyze_missingness, drought_mask, linear_fill, spatial_fill
from src.models.regression import ForecastModel

configured = os.environ.get("ECOLLATERAL_RESEARCH_DIR")
artifact_dir = Path(configured) if configured else root / "data/processed/demo"
if not configured and not (artifact_dir / "model.joblib").exists():
    create_demo(artifact_dir)

model = ForecastModel.load(artifact_dir / "model.joblib")
stations = pd.read_csv(artifact_dir / "stations.csv", index_col="date", parse_dates=["date"])
coordinates = pd.read_csv(artifact_dir / "coordinates.csv", index_col="station_id")
features = pd.read_csv(artifact_dir / "daily_features.csv", index_col="date", parse_dates=["date"])
target = os.environ.get("ECOLLATERAL_TARGET_STATION", "target")
cooling = os.environ.get("ECOLLATERAL_COOLING_TYPE", "cooling_tower")
output = root / "data/processed/research"
output.mkdir(parents=True, exist_ok=True)
print(f"Artifact directory: {artifact_dir}; model data kind: {model.data_kind}")
print("Default demonstration results are not field validation.")

## Candidate performance on a temporal holdout

Training uses the first 60% of distinct dates, calibration uses the next 20%, and the final
20% is reserved for evaluation. Median imputation is fitted on the training period.
Candidate selection uses calibration RMSE. R² is undefined for a constant dry-cooling
consumption target. A nominal 90% split-conformal interval assumes conditions that may
not hold under time dependence or climate shifts; inspect actual test coverage.

In [ ]:
benchmark_rows = []
for technology, report in model.benchmark.items():
    for name, candidate in report["candidates"].items():
        benchmark_rows.append({"cooling_type": technology, "model": name,
                               "selected": name == report["selected"],
                               **candidate["test_bounded"],
                               "selected_interval90_coverage": report["test_interval90_coverage"]})
display(pd.DataFrame(benchmark_rows))

## Streamflow gap experiment

Both strategies receive identical drought-only masks within each seeded replicate.
Nearest-station filling uses available same-day observations in geographic distance
order. Linear time filling reconstructs interior gaps and carries boundary observations;
it uses later data and is suitable for historical reconstruction, not online forecasts.

The stability flag requires at least 90% of drought forecast perturbations within ±0.05 MGD
and every drought record's empirical central 90% band within that tolerance.

In [ ]:
def predict(frame):
    return np.array([result.predicted_mgd for result in model.predict(frame, cooling)])

result = analyze_missingness(stations, target, coordinates, features, predict,
                             repeats=50, tolerance_mgd=0.05, seed=42)
result.summary.to_csv(output / "missingness_summary.csv", index=False)
result.bands.to_csv(output / "missingness_bands.csv", index=False)
display(result.summary)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), layout="constrained")
for station in stations:
    axes[0].plot(stations.index, stations[station], label=station, linewidth=1)
drought = features.pdsi <= -2
axes[0].fill_between(stations.index, 0, stations.max().max(), where=drought,
                     alpha=0.12, color="darkorange", label="Drought window (PDSI <= -2)")
axes[0].set(ylabel="Streamflow (MGD)", title=f"Station observations — {model.data_kind} model workflow")
axes[0].legend(loc="upper right", fontsize=8)
for strategy, rows in result.summary.groupby("strategy"):
    axes[1].plot(rows.missing_rate * 100, rows.tolerance_coverage, "o-", label=strategy)
axes[1].axhline(0.9, color="firebrick", linestyle="--", label="90% tolerance coverage")
axes[1].set(xlabel="Drought records removed (%)", ylabel="Fraction within ±0.05 MGD",
            ylim=(0, 1.05), xticks=[10, 25, 40])
axes[1].legend(fontsize=8)
fig.savefig(output / "missingness_sensitivity.png", dpi=160)
plt.show()

In [ ]:
subset = result.bands[(result.bands.missing_rate == 0.4)
                      & (result.bands.strategy == "spatial_nearest")]
fig, ax = plt.subplots(figsize=(11, 3.8), layout="constrained")
ax.plot(subset.date, subset.baseline_mgd, color="#184e77", label="Complete-series forecast")
ax.fill_between(subset.date, subset.baseline_mgd - 0.05, subset.baseline_mgd + 0.05,
                color="#b5c99a", alpha=0.3, label="±0.05 MGD tolerance")
ax.fill_between(subset.date, subset.band90_low_mgd, subset.band90_high_mgd,
                color="#f48c06", alpha=0.6, label="Empirical 5th–95th percentile")
ax.set(ylabel="Cooling consumption (MGD)", title="40% drought missingness: spatial filling")
ax.legend(fontsize=8)
fig.savefig(output / "missingness_band90.png", dpi=160)
plt.show()

## Groundwater reconstruction: preserve the units

The required regression feature is streamflow in MGD, not groundwater depth in meters.
Groundwater gaps can be reconstructed with the same algorithms, but converting them into
consumption sensitivity requires an independently calibrated hydrologic relationship.
The following separate toy example checks depth reconstruction in meters; it never
substitutes groundwater depths for MGD in the consumption model.

In [ ]:
toy_dates = pd.date_range("2023-01-01", periods=365)
time = np.arange(365)
depth = 20 + 0.4 * np.sin(time / 30)
groundwater = pd.DataFrame({"well": depth,
                            "near_well": depth + 0.02 * np.sin(time / 7)}, index=toy_dates)
well_coords = pd.DataFrame({"latitude": [38.9, 38.91], "longitude": [-77.1, -77.11]},
                           index=groundwater.columns)
well_drought = pd.Series((time >= 150) & (time <= 270), index=toy_dates)
rng = np.random.default_rng(42)
groundwater_rows = []
for rate in (0.1, 0.25, 0.4):
    mask = drought_mask(groundwater.well, well_drought, rate, rng)
    masked = groundwater.copy()
    masked.loc[mask, "well"] = np.nan
    for name, filled in {"spatial_nearest": spatial_fill(masked, "well", well_coords),
                         "linear_time": linear_fill(masked.well)}.items():
        error = filled[mask] - groundwater.well[mask]
        groundwater_rows.append({"missing_rate": rate, "strategy": name,
                                 "masked_records": int(mask.sum()),
                                 "reconstruction_mae_m": float(error.abs().mean())})
groundwater_report = pd.DataFrame(groundwater_rows)
groundwater_report.to_csv(output / "groundwater_reconstruction.csv", index=False)
display(groundwater_report)

## Interpretation and next measurements

- Low perturbation variance means the fitted model changed little under these masks;
  it does not establish that the model's predictions are accurate.
- The synthetic neighbors are strongly correlated. Real droughts can make nearby stations
  fail together or diverge by geology, regulation, or upstream withdrawals.
- The experiment masks random records inside drought windows. Contiguous outages and
  missing-not-at-random extremes need separate experiments.
- An observed study needs complete reference periods, representative adjacent stations,
  station quality review, measured facility targets, and held-out drought/facility validation.
- See `docs/implementation-notes.md` for assumptions, source checks, and validation results.

References: [USGS daily observations](https://api.waterdata.usgs.gov/docs/ogcapi/),
[NOAA nClimDiv](https://www.ncei.noaa.gov/monitoring-content/data/us/climdiv/monthly/current/).